Reading the csv file

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("Data/lung_cancer_mortality_data_large_v2.csv")
df.info()

In [ ]:
import pandas as pd

# Load the first 100,000 rows from the CSV file
df = pd.read_csv("lung_cancer_mortality_data_large_v2.csv", nrows=100000)

# Identify categorical columns
categorical_columns = ['gender', 'country', 'cancer_stage', 'family_history', 
                       'smoking_status', 'treatment_type']

# Perform one-hot encoding on categorical columns
df_encoded = pd.get_dummies(df, columns=categorical_columns)

# Check the result
df_encoded.info()

# Optionally, you can save the result to a new CSV file
#df_encoded.to_csv("lung_cancer_mortality_encoded.csv", index=False)


Applying feature engineering

In [ ]:
import pandas as pd
import featuretools as ft

# Step 1: Load the first 100,000 rows from the CSV file
df = pd.read_csv("lung_cancer_mortality_data_large_v2.csv", nrows=700000)

# Step 2: Identify categorical columns to perform one-hot encoding
categorical_columns = ['gender', 'country', 'cancer_stage', 'family_history', 
                       'smoking_status', 'treatment_type']

# Step 3: Perform one-hot encoding on the specified categorical columns
df_encoded = pd.get_dummies(df, columns=categorical_columns)

# Step 4: Apply Featuretools for automated feature engineering

# Create an EntitySet for Featuretools
es = ft.EntitySet(id="lung_cancer_data")

# Add the dataframe to the EntitySet using `add_dataframe` method
es = es.add_dataframe(dataframe_name="mortality", dataframe=df_encoded, index="id")

# Step 5: Define compatible aggregation and transformation primitives to increase features

# Aggregation primitives
agg_primitives = ["count", "mean", "mode", "sum", "min", "max", "std", "median", 
                  "skew", "num_true"]

# Transformation primitives
trans_primitives = ["multiply_numeric", "divide_numeric", "time_since", "day", 
                    "month", "year", "add_numeric", "subtract_numeric", 
                    "absolute", "percentile", "is_weekend", "is_leap_year"]

# Run DFS to generate new features using the extended primitives and set max_depth for deeper features
feature_matrix, feature_defs = ft.dfs(entityset=es, 
                                      target_dataframe_name="mortality", 
                                      agg_primitives=agg_primitives,
                                      trans_primitives=trans_primitives,
                                      max_features=250,  # Limit to 250 features
                                      max_depth=2)       # Increase depth for more complex features

# Step 6: Ensure 'survived' is the target variable
# Remove 'survived' from the feature matrix to prevent data leakage during feature generation
if 'survived' in feature_matrix.columns:
    feature_matrix = feature_matrix.drop('survived', axis=1)

# Add the original 'survived' column back to the feature matrix
feature_matrix['survived'] = df_encoded['survived']

# Step 7: Save the final feature matrix to a CSV file
feature_matrix.to_csv("lung_cancer_mortality_featured.csv", index=False)

# Check the new feature matrix structure
print(feature_matrix.info())
